# E014 — Audio new augmentation ablation

Tests 4 new augmentations on top of the E008 +All baseline (original + noise SNR=20dB + speed ±10%).

**Configs tested:**
1. `+All` (E008 baseline — re-run for consistency)
2. `+All+Codec` — baseline + 8kHz codec simulation
3. `+All+LowNoise` — baseline + aggressive noise SNR=10dB
4. `+All+Pitch` — baseline + pitch shift ±{1,2} semitones
5. `+All+Clip` — baseline + amplitude clipping at random threshold [30–70% of max]
6. `+All+AllNew` — baseline + all four new augmentations combined

Everything else is identical to E008: UBM 32, MAP r=16, MFCC 13+Δ+ΔΔ+CMN, LOSO seed=67.
Val fold always uses **original WAVs only**.

In [ ]:
from pathlib import Path
import sys, copy
sys.path.insert(0, str(Path("../src").resolve()))

import numpy as np
import librosa
import matplotlib.pyplot as plt
from sklearn.mixture import GaussianMixture
from sklearn.metrics import roc_curve
from scipy.special import logsumexp
from scipy.stats import norm as scipy_norm
import pandas as pd

from data.splits import load_manifest, iter_folds_loso
from eval.metrics import compute_eer, compute_min_dcf

COLORS = {
    "target":    "#E74C3C",
    "nontarget": "#2E86AB",
    "green":     "#27AE60",
    "purple":    "#8E44AD",
    "gray":      "#95A5A6",
    "orange":    "#E67E22",
}
CONFIG_COLORS = {
    "+All (E008)": "#95A5A6",
    "+All+Codec":  "#2E86AB",
    "+All+LowNoise": "#E67E22",
    "+All+Pitch":  "#8E44AD",
    "+All+Clip":   "#27AE60",
    "+All+AllNew": "#E74C3C",
}
plt.rcParams.update({
    "figure.dpi": 130,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
})

DATA = Path("../data").resolve()
manifest = load_manifest(DATA)
y_all = manifest["label"].to_numpy()
SEED = 67
print(f"{len(manifest)} samples — {manifest.label.sum()} target, {(manifest.label==0).sum()} non-target")

## 1. Augmentation functions

E008 +All base augmentations plus four new ones:
- **Codec**: 8kHz resample down + back up (bandwidth limitation, quantization noise)
- **LowNoise**: SNR=10dB additive white noise (more aggressive than E008's 20dB)
- **Pitch**: ±{1,2} semitones random shift (frequency response variation)
- **Clip**: Amplitude clipping at [30–70%] of peak (ADC saturation simulation)

In [ ]:
def find_wav(stem: str, data_dir: Path) -> Path:
    for sf in ["target_train", "target_dev", "non_target_train", "non_target_dev"]:
        p = data_dir / sf / (stem + ".wav")
        if p.exists():
            return p
    raise FileNotFoundError(stem)


# ── E008 base augmentations ──────────────────────────────────────────────────
def aug_noise_20(y: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Additive white noise at SNR=20dB (E008 baseline)."""
    p = np.mean(y ** 2) + 1e-10
    return y + rng.normal(0, np.sqrt(p / 10 ** (20 / 10)), len(y)).astype(y.dtype)


def aug_speed(y: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Random time stretch rate ∈ [0.9, 1.1] (E008 baseline)."""
    return librosa.effects.time_stretch(y, rate=rng.uniform(0.9, 1.1))


# ── New augmentations (E014) ─────────────────────────────────────────────────
def aug_codec(y: np.ndarray, sr: int) -> np.ndarray:
    """Codec simulation: resample to 8kHz and back (bandwidth + quantization)."""
    y8 = librosa.resample(y, orig_sr=sr, target_sr=8000)
    return librosa.resample(y8, orig_sr=8000, target_sr=sr).astype(y.dtype)


def aug_low_noise(y: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Additive white noise at SNR=10dB (more aggressive)."""
    p = np.mean(y ** 2) + 1e-10
    return y + rng.normal(0, np.sqrt(p / 10 ** (10 / 10)), len(y)).astype(y.dtype)


def aug_pitch(y: np.ndarray, sr: int, rng: np.random.Generator) -> np.ndarray:
    """Random pitch shift ±{1,2} semitones."""
    n_steps = float(rng.choice([-2, -1, 1, 2]))
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)


def aug_clip(y: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Amplitude clipping at random threshold [30–70% of peak]."""
    threshold = rng.uniform(0.3, 0.7) * np.max(np.abs(y) + 1e-10)
    return np.clip(y, -threshold, threshold)


# ── MFCC extraction ───────────────────────────────────────────────────────────
def extract_mfcc(y: np.ndarray, sr: int, n_mfcc: int = 13) -> np.ndarray:
    """MFCC + delta + delta-delta + CMN. Returns (T, 39)."""
    mfcc   = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta  = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    mfcc   = np.vstack([mfcc, delta, delta2]).T
    mfcc  -= mfcc.mean(axis=0)  # CMN
    return mfcc


# ── Per-config extract_batch ───────────────────────────────────────────────────
def load_and_augment(wav_path: Path, config: str, rng: np.random.Generator):
    """
    Load WAV, return list of (y, sr) for all copies:
      all:        [orig, noise20, speed]
      +codec:     [orig, noise20, speed, codec]
      +lownoise:  [orig, noise20, speed, noise10]
      +pitch:     [orig, noise20, speed, pitch]
      +clip:      [orig, noise20, speed, clip]
      +allnew:    [orig, noise20, speed, codec, noise10, pitch, clip]
    """
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    samples = [
        (y, sr),
        (aug_noise_20(y, rng), sr),
        (aug_speed(y, rng), sr),
    ]

    if config in ("codec", "allnew"):
        samples.append((aug_codec(y, sr), sr))
    if config in ("lownoise", "allnew"):
        samples.append((aug_low_noise(y, rng), sr))
    if config in ("pitch", "allnew"):
        samples.append((aug_pitch(y, sr, rng), sr))
    if config in ("clip", "allnew"):
        samples.append((aug_clip(y, rng), sr))

    return samples


def extract_batch(df: pd.DataFrame, data_dir: Path, config: str, seed: int):
    """Extract MFCC frames for all samples (with augmentation for train fold)."""
    rng = np.random.default_rng(seed)
    all_mfcc, all_labels = [], []
    for _, row in df.iterrows():
        for y_aug, sr in load_and_augment(find_wav(row["stem"], data_dir), config, rng):
            mfcc = extract_mfcc(y_aug, sr)
            all_mfcc.append(mfcc)
            all_labels.extend([row["label"]] * len(mfcc))
    return np.vstack(all_mfcc), np.array(all_labels)


def extract_batch_original(df: pd.DataFrame, data_dir: Path):
    """Extract MFCC from original WAVs only (for val fold)."""
    all_mfcc, all_labels = [], []
    for _, row in df.iterrows():
        y, sr = librosa.load(find_wav(row["stem"], data_dir), sr=None, mono=True)
        mfcc = extract_mfcc(y, sr)
        all_mfcc.append(mfcc)
        all_labels.extend([row["label"]] * len(mfcc))
    return np.vstack(all_mfcc), np.array(all_labels)


print("Augmentation functions defined.")

## 2. Augmentation visualization

Visualize all six augmented versions of one target utterance to confirm they sound
realistic — they should resemble the same person under different recording conditions.

In [ ]:
rng_viz = np.random.default_rng(42)
sample_row = manifest[manifest.label == 1].iloc[0]
y_orig, sr = librosa.load(find_wav(sample_row["stem"], DATA), sr=None, mono=True)

signals = {
    "Original": y_orig,
    "+Noise 20dB (E008)": aug_noise_20(y_orig, rng_viz),
    "+Speed (E008)": aug_speed(y_orig, rng_viz),
    "+Codec (8kHz sim)": aug_codec(y_orig, sr),
    "+LowNoise 10dB": aug_low_noise(y_orig, rng_viz),
    "+Pitch (±semitone)": aug_pitch(y_orig, sr, rng_viz),
    "+Clip (30–70%)": aug_clip(y_orig, rng_viz),
}

n = len(signals)
fig, axes = plt.subplots(n, 2, figsize=(14, 3 * n))

for row_idx, (title, y_aug) in enumerate(signals.items()):
    t = np.linspace(0, len(y_aug) / sr, len(y_aug))

    ax = axes[row_idx, 0]
    ax.plot(t, y_aug, color=COLORS["nontarget"], lw=0.4, alpha=0.8)
    ax.set_title(f"{title} — waveform", fontsize=9)
    ax.set_xlabel("Time [s]")
    ax.set_ylabel("Amplitude")
    ax.set_xlim(0, max(t))

    ax = axes[row_idx, 1]
    mfcc_aug = extract_mfcc(y_aug, sr)[:, :13]
    ax.imshow(mfcc_aug.T, aspect="auto", origin="lower", cmap="magma")
    ax.set_title(f"{title} — MFCC (static 13)", fontsize=9)
    ax.set_xlabel("Frame")
    ax.set_ylabel("MFCC coeff")

plt.suptitle(f"E014 — Audio augmentations — {sample_row['stem']} (target)",
             color=COLORS["target"], fontsize=12)
plt.tight_layout()
plt.show()

## 3. UBM + MAP functions (identical to E008)

In [ ]:
def train_ubm(X: np.ndarray, n_components: int = 32, seed: int = 67) -> GaussianMixture:
    return GaussianMixture(
        n_components=n_components, covariance_type="diag",
        max_iter=200, random_state=seed,
    ).fit(X)


def map_adapt(ubm: GaussianMixture, X_target: np.ndarray, r: float = 16.0) -> GaussianMixture:
    log_prob  = ubm._estimate_log_prob(X_target)
    log_resp  = log_prob + np.log(ubm.weights_)
    log_resp -= logsumexp(log_resp, axis=1, keepdims=True)
    resp      = np.exp(log_resp)
    n_k       = resp.sum(axis=0)
    mu_hat    = (resp.T @ X_target) / (n_k[:, None] + 1e-10)
    alpha     = n_k / (n_k + r)
    adapted   = copy.deepcopy(ubm)
    adapted.means_ = alpha[:, None] * mu_hat + (1 - alpha[:, None]) * ubm.means_
    return adapted


def score_utterance(wav_path: Path, adapted: GaussianMixture, ubm: GaussianMixture) -> float:
    y, sr = librosa.load(wav_path, sr=None, mono=True)
    mfcc  = extract_mfcc(y, sr)
    return float((adapted.score_samples(mfcc) - ubm.score_samples(mfcc)).mean())


print("UBM+MAP functions defined.")

## 4. Cross-validation across all configs

For each config, UBM and MAP-adapted model are trained on augmented train frames.
Val utterances are always scored from **original WAVs**.
The E008 +All baseline is re-run fresh for a fair within-experiment comparison.

In [ ]:
UBM_COMPONENTS = 32
MAP_R = 16.0

# config_key: internal augmentation key, config_name: display label
CONFIGS = {
    "all":      "+All (E008)",
    "codec":    "+All+Codec",
    "lownoise": "+All+LowNoise",
    "pitch":    "+All+Pitch",
    "clip":     "+All+Clip",
    "allnew":   "+All+AllNew",
}

all_results = {}
all_oof     = {}

for config_key, config_name in CONFIGS.items():
    print(f"\n{'='*55}")
    print(f"Config: {config_name}")
    print('=' * 55)

    oof_scores   = np.full(len(manifest), np.nan)
    fold_results = []

    for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
        train_df = manifest.loc[train_idx]
        val_df   = manifest.loc[val_idx]

        # Augmented train frames (only train fold!)
        X_train, y_train = extract_batch(train_df, DATA, config_key, seed=SEED + fold_id)
        X_nt = X_train[y_train == 0]
        X_t  = X_train[y_train == 1]

        print(f"  Fold {fold_id}: {len(X_t):,} target frames, {len(X_nt):,} non-target frames (aug)")

        ubm     = train_ubm(X_nt, n_components=UBM_COMPONENTS, seed=SEED)
        adapted = map_adapt(ubm, X_t, r=MAP_R)

        # Score val on ORIGINAL WAVs only
        for idx, row in val_df.iterrows():
            oof_scores[idx] = score_utterance(find_wav(row["stem"], DATA), adapted, ubm)

        val_scores = oof_scores[val_idx]
        val_labels = manifest.loc[val_idx, "label"].to_numpy()
        eer, _     = compute_eer(val_scores[val_labels == 1], val_scores[val_labels == 0])
        min_dcf, _ = compute_min_dcf(val_scores[val_labels == 1], val_scores[val_labels == 0])
        fold_results.append({"fold": fold_id, "eer": eer, "min_dcf": min_dcf})
        print(f"  → EER={eer*100:.2f}%, min-DCF={min_dcf:.4f}")

    all_results[config_name] = fold_results
    all_oof[config_name]     = oof_scores.copy()

print("\nAll configs done.")

## 5. Results — ablation table

In [ ]:
print(f"{'Config':<18} {'F0 EER':>8} {'F1 EER':>8} {'F2 EER':>8} {'Mean':>8} {'Std':>8} {'min-DCF':>9}")
print("-" * 72)

summary = []
for config_name, fold_results in all_results.items():
    eers   = [r["eer"] * 100 for r in fold_results]
    dcfs   = [r["min_dcf"]   for r in fold_results]
    mean_e = np.mean(eers)
    std_e  = np.std(eers)
    mean_d = np.mean(dcfs)
    marker = "" if config_name == "+All (E008)" else ""
    print(f"{config_name:<18} {eers[0]:>8.2f} {eers[1]:>8.2f} {eers[2]:>8.2f} "
          f"{mean_e:>8.2f} {std_e:>8.2f} {mean_d:>9.4f}")
    summary.append({"config": config_name, "f0": eers[0], "f1": eers[1], "f2": eers[2],
                    "mean": mean_e, "std": std_e, "min_dcf": mean_d})

print("-" * 72)

# Exclude E008 baseline from best-selection (it's the reference)
new_configs = [s for s in summary if s["config"] != "+All (E008)"]
best = min(new_configs, key=lambda x: x["mean"])
best_overall = min(summary, key=lambda x: x["mean"])

e008_ref = next(s for s in summary if s["config"] == "+All (E008)")
print(f"\nE008 reference (+All): EER={e008_ref['mean']:.2f}±{e008_ref['std']:.2f}%")
print(f"Best new config: {best['config']}  EER={best['mean']:.2f}±{best['std']:.2f}%")
print(f"Best overall:    {best_overall['config']}  EER={best_overall['mean']:.2f}±{best_overall['std']:.2f}%")

oof_best_overall = all_oof[best_overall["config"]]
eer_oof, _   = compute_eer(oof_best_overall[y_all == 1], oof_best_overall[y_all == 0])
dcf_oof, thr = compute_min_dcf(oof_best_overall[y_all == 1], oof_best_overall[y_all == 0])
print(f"OOF overall ({best_overall['config']}): EER={eer_oof*100:.2f}%, min-DCF={dcf_oof:.4f}, threshold={thr:.3f}")

## 6. Visualizations

Three plots:
1. Mean EER ± std bar chart (with E008 baseline as reference)
2. DET curves overlay for all 6 configs (OOF scores)
3. Per-fold grouped bar chart

In [ ]:
configs_list = list(all_results.keys())
n_configs = len(configs_list)
means  = [np.mean([r["eer"] * 100 for r in all_results[c]]) for c in configs_list]
stds   = [np.std( [r["eer"] * 100 for r in all_results[c]]) for c in configs_list]
colors = [CONFIG_COLORS[c] for c in configs_list]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(range(n_configs), means, color=colors, alpha=0.85,
              yerr=stds, capsize=7, error_kw=dict(elinewidth=2))

# E008 reference dashed line
ax.axhline(e008_ref["mean"], color="#95A5A6", ls="--", lw=1.5,
           label=f"E008 +All reference ({e008_ref['mean']:.2f}%)")

for bar, m, s in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width() / 2, m + s + 0.08,
            f"{m:.2f}\n±{s:.2f}", ha="center", fontsize=8)

# Highlight best
best_idx_overall = np.argmin(means)
bars[best_idx_overall].set_edgecolor("gold")
bars[best_idx_overall].set_linewidth(3)
ax.annotate("★ best",
            xy=(best_idx_overall, means[best_idx_overall] - stds[best_idx_overall] - 0.3),
            ha="center", fontsize=9, color="goldenrod", fontweight="bold")

# E008 baseline stripe (first bar)
bars[0].set_hatch("//")
bars[0].set_edgecolor("#666666")

ax.set_xticks(range(n_configs))
ax.set_xticklabels(configs_list, fontsize=9)
ax.set_ylabel("EER mean ± std [%]")
ax.set_title("E014 — Mean EER per config (+All E008 baseline + new augmentations)", fontsize=12)
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
ticks = [0.01, 0.05, 0.1, 0.2, 0.3, 0.4]
tick_pos    = [scipy_norm.ppf(t) for t in ticks]
tick_labels = [f"{int(t * 100)}" for t in ticks]

fig, ax = plt.subplots(figsize=(7, 6))
for config_name, oof_s in all_oof.items():
    valid = ~np.isnan(oof_s)
    fpr, tpr, _ = roc_curve(y_all[valid], oof_s[valid])
    far_c = np.clip(fpr, 1e-4, 1 - 1e-4)
    frr_c = np.clip(1 - tpr, 1e-4, 1 - 1e-4)
    eer_c, _ = compute_eer(oof_s[valid & (y_all == 1)], oof_s[valid & (y_all == 0)])
    lw = 2.5 if config_name == best_overall["config"] else 1.5
    ls = "--" if config_name == "+All (E008)" else "-"
    ax.plot(scipy_norm.ppf(far_c), scipy_norm.ppf(frr_c),
            color=CONFIG_COLORS[config_name], lw=lw, ls=ls,
            label=f"{config_name}  EER={eer_c*100:.1f}%",
            zorder=5 if config_name == best_overall["config"] else 1)

ax.plot(tick_pos, tick_pos, "k--", lw=1)
ax.set_xticks(tick_pos); ax.set_xticklabels(tick_labels)
ax.set_yticks(tick_pos); ax.set_yticklabels(tick_labels)
ax.set_xlabel("FAR [%]")
ax.set_ylabel("FRR [%]")
ax.set_title("E014 — DET curves (OOF scores, all configs)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(3)
width = 0.13

for i, config_name in enumerate(configs_list):
    eers = [r["eer"] * 100 for r in all_results[config_name]]
    offset = (i - n_configs / 2 + 0.5) * width
    hatch = "//" if config_name == "+All (E008)" else None
    ax.bar(x + offset, eers, width, label=config_name,
           color=CONFIG_COLORS[config_name], alpha=0.85, hatch=hatch)

ax.set_xticks(x)
ax.set_xticklabels(["Fold 0 (session 01)", "Fold 1 (session 02)", "Fold 2 (session 03)"])
ax.set_ylabel("EER [%]")
ax.set_title("E014 — Per-fold EER across all augmentation configs")
ax.legend(fontsize=8, ncol=3)
plt.tight_layout()
plt.show()

In [ ]:
# Score distribution for the best config (OOF)
oof_best_arr = all_oof[best_overall["config"]]
eer_oof_b, _    = compute_eer(oof_best_arr[y_all == 1], oof_best_arr[y_all == 0])
dcf_oof_b, thr_b = compute_min_dcf(oof_best_arr[y_all == 1], oof_best_arr[y_all == 0])

fold_scores_best = []
for fold_id, train_idx, val_idx in iter_folds_loso(manifest, seed=SEED):
    fold_scores_best.append({
        "scores": oof_best_arr[val_idx],
        "labels": manifest.loc[val_idx, "label"].to_numpy()
    })

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
bin_edges = np.linspace(np.nanmin(oof_best_arr), np.nanmax(oof_best_arr), 35)

for i, (ax, fdata) in enumerate(zip(axes[:3], fold_scores_best)):
    s, l = fdata["scores"], fdata["labels"]
    eer_f, thr_f = compute_eer(s[l == 1], s[l == 0])
    ax.hist(s[l == 0], bins=bin_edges, alpha=0.6, color=COLORS["nontarget"], label="non-target", density=True)
    ax.hist(s[l == 1], bins=bin_edges, alpha=0.75, color=COLORS["target"], label="target", density=True)
    ax.axvline(thr_f, color=COLORS["green"], ls="--", lw=2, label=f"thresh={thr_f:.2f}")
    ax.set_title(f"Fold {i}  —  EER={eer_f*100:.1f}%")
    ax.set_xlabel("Score (LLR)")
    ax.legend(fontsize=8)

ax = axes[3]
ax.hist(oof_best_arr[y_all == 0], bins=bin_edges, alpha=0.6, color=COLORS["nontarget"], label="non-target", density=True)
ax.hist(oof_best_arr[y_all == 1], bins=bin_edges, alpha=0.75, color=COLORS["target"], label="target", density=True)
ax.axvline(thr_b, color=COLORS["green"], ls="--", lw=2, label=f"OOF thresh={thr_b:.2f}")
ax.set_title(f"OOF overall  —  EER={eer_oof_b*100:.1f}%")
ax.set_xlabel("Score (LLR)")
ax.legend(fontsize=8)

plt.suptitle(f"E014 — Score distributions, best config: {best_overall['config']}", fontsize=12)
plt.tight_layout()
plt.show()

print(f"E008 +All reference: EER=4.21±3.11%")
print(f"E014 best config ({best_overall['config']}): EER={best_overall['mean']:.2f}±{best_overall['std']:.2f}%")
print(f"OOF threshold: {thr_b:.3f}")